In [1]:
import argparse as arg
import os
from datetime import datetime

import pandas as pd
import yaml
from torch.cuda import is_available
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

from inference.inference import Inference
from postprocess.postprocess import PostProcess
from preprocess.preprocess import SectionRemover, Preprocess

with open("config.yaml") as f:
    params = yaml.safe_load(f)
    

device = "cuda:0"
additional_rules = None

include_cols = params["pre_process"]["include_cols"]
#include_cols.append(params["pre_process"]["line_col"])

section_remover_for_inference = SectionRemover(lang_model=os.path.abspath(params["pre_process"]["lang_model"]),
                                               remove_sections=params["pre_process"]["remove_sections"],
                                               keep_sections=params["pre_process"]["inference_sections"],
                                               rules_json=params["pre_process"]["section_rules"],
                                               additional_rules=additional_rules,
                                               gpu=device)

In [10]:
files=["test.xlsx"]

In [13]:
#files=os.listdir("to_run")
for file in files:
    #notes="to_run/{}".format(file)
    notes=file
    output=file.replace(".xlsx", "_report.xlsx")
    if os.path.exists(output) or "report" in notes:
        print("{} exitst".format(output))
    else:
        print("processing {}".format(notes))
        preprocessed_notes = Preprocess(notes, params["pre_process"]["terms_to_fix"])
        preprocessed_notes = preprocessed_notes.read_raw_notes()
        #preprocessed_notes.raw_notes["LOS"]=1
        additional_columns = params["pre_process"]["include_cols"] + [params["pre_process"]["line_col"]]
        print("preprocess")
        preprocessed_notes = preprocessed_notes.get_relevant_notes(filters=params["pre_process"]["note_types"],
                                                                   additional_columns=additional_columns)
        
        preprocessed_notes = preprocessed_notes.merge_notes(section_remover=section_remover_for_inference,
                                                        include_cols=include_cols,
                                                        group_cols=params["pre_process"]["group_cols"],
                                                        orientation=params["pre_process"]["orientation"],
                                                        keep_unlabelled=params["pre_process"]["keep_unlabelled"],
                                                        anonymize=params["pre_process"]["anonymize"],
                                                        language_model=params["pre_process"]["lang_model"],
                                                        line_col=params["pre_process"]["line_col"])
        print("inference")
        inference_notes = preprocessed_notes.merged_raw.copy()
        inference_notes = inference_notes[~pd.isnull(inference_notes[params["inference"]["note_col"]])].copy()
        infer_notes = Inference(classification_model=os.path.abspath(params["inference"]["classification_model"]),
                            summarization_model=os.path.abspath(params["inference"]["summarization_model"]),
                            classification_labels=params["inference"]["classification_labels"],
                            intent_model=os.path.abspath(params["inference"]["intent_model"]),
                            intent_labels=params["inference"]["intent_labels"],
                            substance_model=os.path.abspath(params["inference"]["substance_model"]),
                            substance_labels=params["inference"]["substance_labels"],
                            io_model=os.path.abspath(params["inference"]["io_model"]),
                            io_labels=params["inference"]["io_labels"],
                            device=device)
        probs = infer_notes.classify(inference_notes,
                                 params["inference"]["note_col"],
                                 params["inference"]["include_labels"])
        inference_notes["probs"] = probs
        inference_notes = inference_notes.sort_values(by=["probs"], ascending=False)

        complaint_filter = inference_notes["Chief Complaint"].isin(params["inference"]["pos_complaints"])
        pos_based_on_complaint = inference_notes[complaint_filter]
        num_rows = pos_based_on_complaint.shape[0]
        row_cutoff = round(num_rows * params["inference"]["cutoff"]) - 1
        prob_cutoff = pos_based_on_complaint["probs"].tolist()[row_cutoff]
        inference_notes["probs"][complaint_filter] = 1
        inference_notes["probs"][inference_notes[params["inference"]["chirpp_col"]] == "CHIRPP ICON"] = 1
        inference_notes["to_summarize"] = inference_notes["probs"] >= prob_cutoff
        print("summarize")
        summaries = infer_notes.summarize(inference_notes[inference_notes["to_summarize"]],
                                      params["inference"]["note_col"],
                                      params["inference"]["truncation"],
                                      params["inference"]["max_length"])
        inference_notes["PHAC Narrative"]=None
        inference_notes["cosine_similarity"]=None
        inference_notes["PHAC Narrative"][inference_notes["to_summarize"]] = summaries
        print("intent")
        intent = infer_notes.get_intent(notes=inference_notes[inference_notes["to_summarize"]],
                                    notes_col=params["inference"]["note_col"],
                                    label_dict=params["inference"]["intent_label_dict"],
                                    cutoff=params["inference"]["intent_cutoff"])
        inference_notes["intent"] = None
        inference_notes["intent"][inference_notes["to_summarize"]] = intent
        print("substance")
        substance = infer_notes.get_substance(notes=inference_notes[inference_notes["to_summarize"]],
                                      notes_col=params["inference"]["note_col"],
                                      cutoff=params["inference"]["subs_cutoff"])
        inference_notes["sub"] = None
        inference_notes["sub"][inference_notes["to_summarize"]] = substance
        print("io")
        io = infer_notes.get_io(notes=inference_notes[(inference_notes["to_summarize"]) & (inference_notes["intent"] == 10)],
                            notes_col=params["inference"]["note_col"],
                            cutoff=params["inference"]["io_cutoff"])
        inference_notes["io"] = None
        inference_notes["io"][(inference_notes["to_summarize"]) & (inference_notes["intent"] == 10)] = io
        params["post_process"]["pos_complaints"] = params["inference"]["pos_complaints"]
        postprocess = PostProcess(preprocessed_notes.raw_notes, inference_notes, params["post_process"])
        print("autofill")
        postprocess = postprocess.autofill()
        postprocess.create_report(output)

processing test.xlsx
preprocess
inference
summarize
intent
substance
io
autofill


In [13]:
postprocess = PostProcess(preprocessed_notes.raw_notes, inference_notes, params["post_process"])
print("autofill")
postprocess = postprocess.autofill()
postprocess.create_report(output)

autofill


In [12]:
postprocess.sheet1.columns

Index(['MRN', 'ScrMRN', 'DOB', 'SEX', 'POSTAL', 'ER Time', 'ER Date',
       'INJ DATE', 'Hr', 'Min', 'AM/PM', 'I/O', 'LOCATION', 'AREA', 'PLACE',
       'Diagnosis', 'SK Narrative', 'PHAC Narrative', 'W4P', 'NO1', 'BP1',
       'NO2', 'veh', 'veh p', 'BP2', 'NO3', 'BP3', 'Notes', 'LOS', 'DISP',
       'IN', 'sub', 'subID', 'sd1', 'sd2', 'sd3', 'sd4', 'sd5', 'SPORTS CODE',
       'E1', 'E2', 'E3', 'E4', 'CTAS', 'Chief Complaint', 'Problem List',
       'probs', 'pre_processed', 'Disposition', 'to_summarize'],
      dtype='object')

In [12]:
import os

import medspacy
import spacy
from medspacy.context import ConText

from postprocess.extra_contex_rules import context_rules
from postprocess.target_rules import *

parse_nlp = spacy.load('en_core_web_trf', disable=["ner"])
med_nlp = medspacy.load(medspacy_enable=['medspacy_sectionizer'])
target = med_nlp.add_pipe("medspacy_target_matcher")

context = ConText(med_nlp, rules='postprocess/context_rules.json')
context.add(context_rules)
target.add(substances)


In [13]:
text="Around 1830-1900 pt was playing in kitchenette and parents found a bottle of advil opened with 2 tablets (400mg) on the floor. Bottle was open without safety lock."

In [15]:
doc = med_nlp(str(text))

In [19]:
substance = ",".join(list(set([str(ent).lower() for ent in doc.ents])))
substance

'advil'

In [6]:
postprocess.create_report(output)

In [7]:
include_cols=list(set(include_cols))

In [10]:
postprocess = postprocess.autofill()
postprocess.create_report(output)

KeyError: 'pre_processed'

In [6]:
files

['Apr_2023.xlsx',
 'Dec_2022.xlsx',
 'Dec_2023.xlsx',
 'Feb_2020.xlsx',
 'Feb_2022.xlsx',
 'Feb_2023.xlsx',
 'Jan_2020.xlsx',
 'Jan_2022.xlsx',
 'Jan_2023.xlsx',
 'Mar_2023.xlsx',
 'Nov_2022.xlsx']

In [9]:
files.sort()
files

['Apr_2023.xlsx',
 'Dec_2022.xlsx',
 'Dec_2023.xlsx',
 'Feb_2020.xlsx',
 'Feb_2022.xlsx',
 'Feb_2023.xlsx',
 'Jan_2020.xlsx',
 'Jan_2022.xlsx',
 'Jan_2023.xlsx',
 'Mar_2023.xlsx',
 'Nov_2022.xlsx']